# Validate Cropland Outputs

This notebook starts after `02_explore_cropland_mask.ipynb` has produced January-June 2026 cropland rasters and field parcel polygons.

The goal is to check whether the cropland candidate layer is usable before it becomes an MRV input. We do that by comparing the outputs against the Landsat mosaics, calculating area summaries, filtering obvious polygon noise, and creating validation sample points for later manual or field review.


## Workflow

1. **Load notebook 02 outputs**: cropland pixel raster, field parcel raster, field parcel polygons, AOI boundary, and Landsat mosaics.
2. **Visual QA**: overlay cropland pixels and parcels on true-color and false-color Landsat composites.
3. **Area accounting**: calculate AOI area, cropland pixel area, parcel polygon area, cropland share, parcel count, and parcel size distribution.
4. **Filter noisy parcels**: remove very small polygons and flag very large polygons for review.
5. **Write cleaned outputs**: export cleaned parcel polygons and a cleaned parcel raster aligned to the Landsat/cropland grid.
6. **Create validation samples**: generate random points for cropland parcels, non-cropland AOI, and parcel-edge/uncertain areas.
7. **Export QA tables**: write summary CSVs and validation points so the results can be reviewed outside the notebook.

This is not final accuracy assessment yet. It prepares the data we need for accuracy assessment: clean candidate outputs and places to inspect.


In [ ]:
from pathlib import Path
import gc
import os
import sys

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import rasterio
from rasterio.features import rasterize
from rasterio.plot import plotting_extent
from shapely.geometry import Point
from shapely.ops import unary_union

def mount_google_drive_if_colab() -> None:
    try:
        from google.colab import drive
    except ModuleNotFoundError:
        return

    drive.mount("/content/drive")


mount_google_drive_if_colab()

COLAB_PROJECT_ROOT = Path("/content/drive/MyDrive/erw_spatial_mrv")
COLAB_DATA_ROOT = COLAB_PROJECT_ROOT / "data"
LOCAL_PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()


def has_erw_package(project_root: Path) -> bool:
    return (project_root / "src" / "erw_mrv" / "__init__.py").exists()


def source_root_candidates() -> list[Path]:
    cwd = Path.cwd().resolve()
    candidates = [LOCAL_PROJECT_ROOT, COLAB_PROJECT_ROOT]
    for base in (cwd, *cwd.parents):
        candidates.extend((base, base / "erw_spatial_mrv"))
    candidates.extend(
        Path(path)
        for path in (
            "/content/erw_spatial_mrv",
            "/content/enhanced_rock_weathering/erw_spatial_mrv",
            "/content/drive/MyDrive/erw_spatial_mrv",
        )
    )
    unique = []
    for candidate in candidates:
        if candidate not in unique:
            unique.append(candidate)
    return unique


def find_source_project_root() -> Path:
    for candidate in source_root_candidates():
        if has_erw_package(candidate):
            return candidate
    checked = chr(10).join(f"- {candidate}" for candidate in source_root_candidates())
    raise ModuleNotFoundError(
        "Could not find src/erw_mrv. The data can live in Google Drive, but "
        "the notebook still needs the project source folder containing src/erw_mrv. "
        "In Colab, upload/sync the full erw_spatial_mrv project or run from a "
        "checkout that includes src/. Checked: "
        f"{checked}"
    )


SOURCE_PROJECT_ROOT = find_source_project_root()
PROJECT_ROOT = COLAB_PROJECT_ROOT if COLAB_DATA_ROOT.exists() else SOURCE_PROJECT_ROOT
DATA_ROOT = COLAB_DATA_ROOT if COLAB_DATA_ROOT.exists() else PROJECT_ROOT / "data"
os.environ["ERW_MRV_DATA_ROOT"] = str(DATA_ROOT)

SRC = SOURCE_PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"SOURCE_PROJECT_ROOT = {SOURCE_PROJECT_ROOT}")
print(f"DATA_ROOT = {DATA_ROOT}")

plt.rcParams["figure.figsize"] = (10, 8)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PROJECT_ROOT = /content/drive/MyDrive/erw_spatial_mrv
SOURCE_PROJECT_ROOT = /content/drive/MyDrive/erw_spatial_mrv
DATA_ROOT = /content/drive/MyDrive/erw_spatial_mrv/data


## Configure Inputs And Outputs

These paths assume notebook `02` used the January-June 2026 workflow. If you later change the date window, change `OUTPUT_TAG` and `DATE_TAG` here to match.


In [ ]:
DATE_TAG = "202601_202606"
OUTPUT_TAG = "jan_jun_2026"
SHOW_QA_PLOTS = False

AOI_PATH = PROJECT_ROOT / "data" / "processed" / "boundaries" / "selected_districts_aoi.geojson"
LANDCOVER_DIR = PROJECT_ROOT / "data" / "processed" / "landcover"
MOSAIC_DIR = PROJECT_ROOT / "data" / "processed" / "landsat" / "mosaics" / DATE_TAG
QA_DIR = LANDCOVER_DIR / "validation" / OUTPUT_TAG
QA_DIR.mkdir(parents=True, exist_ok=True)

CROPLAND_PIXELS_PATH = LANDCOVER_DIR / f"cropland_pixels_{OUTPUT_TAG}_paper_adapted.tif"
FIELD_PARCEL_RASTER_PATH = LANDCOVER_DIR / f"field_parcels_{OUTPUT_TAG}_paper_adapted.tif"
FIELD_PARCELS_PATH = LANDCOVER_DIR / f"field_parcels_{OUTPUT_TAG}_paper_adapted.gpkg"

HAS_FIELD_PARCELS = FIELD_PARCEL_RASTER_PATH.exists() and FIELD_PARCELS_PATH.exists()

MOSAIC_PATTERN = "landsat_{band}_mosaic_aoi.tif"
RGB_BANDS = ("red", "green", "blue")
FALSE_COLOR_BANDS = ("nir08", "red", "green")

required_paths = [
    AOI_PATH,
    CROPLAND_PIXELS_PATH,
    *[MOSAIC_DIR / MOSAIC_PATTERN.format(band=band) for band in set(RGB_BANDS + FALSE_COLOR_BANDS)],
]
missing_paths = [path for path in required_paths if not path.exists()]
if missing_paths:
    raise FileNotFoundError(
        "Missing required output(s). Finish notebook 02 first, then rerun this notebook. Missing: "
        + "; ".join(str(path) for path in missing_paths)
    )

print(f"QA outputs will be written to: {QA_DIR}")
print(f"Cropland raster: {CROPLAND_PIXELS_PATH}")
print(f"Field parcel outputs available: {HAS_FIELD_PARCELS}")


QA outputs will be written to: /content/drive/MyDrive/erw_spatial_mrv/data/processed/landcover/validation/jan_jun_2026
Cropland raster: /content/drive/MyDrive/erw_spatial_mrv/data/processed/landcover/cropland_pixels_jan_jun_2026_paper_adapted.tif
Field parcel outputs available: False


## Helper Functions


In [ ]:
SCALE = 0.0000275
OFFSET = -0.2


def read_mask(path):
    with rasterio.open(path) as src:
        data = src.read(1)
        profile = src.profile.copy()
    return data, profile


def read_reflectance(path):
    with rasterio.open(path) as src:
        raw = src.read(1).astype("float32")
        nodata = src.nodata
    invalid = raw <= 0
    if nodata is not None:
        invalid |= raw == nodata
    data = raw * SCALE + OFFSET
    data[invalid] = np.nan
    return data.astype("float32")


def stretch(values, lower=2, upper=98):
    lo, hi = np.nanpercentile(values, [lower, upper])
    if hi <= lo:
        return np.zeros_like(values, dtype="float32")
    return np.clip((values - lo) / (hi - lo), 0, 1)


def read_composite(bands):
    arrays = [read_reflectance(MOSAIC_DIR / MOSAIC_PATTERN.format(band=band)) for band in bands]
    return np.dstack([stretch(array) for array in arrays])


def pixel_area_m2(profile):
    transform = profile["transform"]
    return abs(transform.a * transform.e)


def sample_points_in_geometry(geometry, count, rng, max_attempts=100_000):
    if geometry.is_empty or count <= 0:
        return []
    minx, miny, maxx, maxy = geometry.bounds
    points = []
    attempts = 0
    while len(points) < count and attempts < max_attempts:
        attempts += 1
        point = Point(rng.uniform(minx, maxx), rng.uniform(miny, maxy))
        if geometry.contains(point):
            points.append(point)
    if len(points) < count:
        print(f"Warning: requested {count} points, generated {len(points)} points.")
    return points


def geometry_union(geometries):
    if hasattr(geometries, "union_all"):
        return geometries.union_all()
    return unary_union(list(geometries))


## Load Outputs


In [ ]:
cropland_pixels, crop_profile = read_mask(CROPLAND_PIXELS_PATH)
aoi = gpd.read_file(AOI_PATH).to_crs(crop_profile["crs"])

if HAS_FIELD_PARCELS:
    field_parcel_raster, parcel_profile = read_mask(FIELD_PARCEL_RASTER_PATH)
    field_parcels = gpd.read_file(FIELD_PARCELS_PATH).to_crs(crop_profile["crs"])
    parcel_mask = field_parcel_raster == 1
else:
    field_parcel_raster = None
    parcel_profile = None
    field_parcels = gpd.GeoDataFrame({"value": []}, geometry=[], crs=crop_profile["crs"])
    parcel_mask = None

aoi_geom = geometry_union(aoi.geometry)
raster_extent = plotting_extent(cropland_pixels, crop_profile["transform"])
crop_mask = cropland_pixels == 1

print(f"Cropland pixels: {crop_mask.sum():,}")
if HAS_FIELD_PARCELS:
    print(f"Field parcel pixels: {parcel_mask.sum():,}")
    print(f"Field parcel polygons: {len(field_parcels):,}")
else:
    print("Field parcel QA skipped because notebook 02 ran in cropland-only mode.")


Cropland pixels: 18,631,771
Field parcel QA skipped because notebook 02 ran in cropland-only mode.


## Visual QA

Green shows cropland pixels; magenta outlines show extracted field parcel polygons.


In [ ]:
if SHOW_QA_PLOTS:
    true_color = read_composite(RGB_BANDS)
    false_color = read_composite(FALSE_COLOR_BANDS)

    if HAS_FIELD_PARCELS:
        fig, axes = plt.subplots(1, 2, figsize=(18, 9))

        axes[0].imshow(np.nan_to_num(true_color, nan=0), extent=raster_extent)
        axes[0].imshow(np.where(crop_mask, 1, np.nan), cmap="summer", alpha=0.35, extent=raster_extent)
        field_parcels.boundary.plot(ax=axes[0], color="magenta", linewidth=0.4)
        axes[0].set_title("True color + cropland pixels + parcel boundaries")
        axes[0].set_axis_off()

        axes[1].imshow(np.nan_to_num(false_color, nan=0), extent=raster_extent)
        axes[1].imshow(np.where(parcel_mask, 1, np.nan), cmap="spring", alpha=0.35, extent=raster_extent)
        field_parcels.boundary.plot(ax=axes[1], color="white", linewidth=0.4)
        axes[1].set_title("False color + field parcel raster + parcel boundaries")
        axes[1].set_axis_off()
    else:
        fig, axes = plt.subplots(1, 2, figsize=(18, 9))

        axes[0].imshow(np.nan_to_num(true_color, nan=0), extent=raster_extent)
        axes[0].imshow(np.where(crop_mask, 1, np.nan), cmap="summer", alpha=0.35, extent=raster_extent)
        axes[0].set_title("True color + cropland pixels")
        axes[0].set_axis_off()

        axes[1].imshow(np.nan_to_num(false_color, nan=0), extent=raster_extent)
        axes[1].imshow(np.where(crop_mask, 1, np.nan), cmap="summer", alpha=0.35, extent=raster_extent)
        axes[1].set_title("False color + cropland pixels")
        axes[1].set_axis_off()

    plt.tight_layout()
    plt.show()
    del true_color, false_color
    gc.collect()
else:
    print("Skipping full-raster QA plots. Set SHOW_QA_PLOTS = True to display them.")


Skipping full-raster QA plots. Set SHOW_QA_PLOTS = True to display them.


## Area Summary

This calculates the key accounting numbers we need before using the layer downstream.


In [ ]:
px_area_m2 = pixel_area_m2(crop_profile)
aoi_area_m2 = float(aoi.geometry.area.sum())
cropland_pixel_area_m2 = float(crop_mask.sum() * px_area_m2)

summary_rows = [
    {"metric": "AOI area", "value": aoi_area_m2 / 10_000, "unit": "ha"},
    {"metric": "Cropland pixel area", "value": cropland_pixel_area_m2 / 10_000, "unit": "ha"},
    {"metric": "Cropland pixel share of AOI", "value": cropland_pixel_area_m2 / aoi_area_m2, "unit": "fraction"},
]

if HAS_FIELD_PARCELS:
    parcel_raster_area_m2 = float(parcel_mask.sum() * px_area_m2)
    field_parcels = field_parcels.copy()
    field_parcels["area_m2"] = field_parcels.geometry.area
    field_parcels["area_ha"] = field_parcels["area_m2"] / 10_000
    field_parcels["perimeter_m"] = field_parcels.geometry.length
    field_parcels["compactness"] = (4 * np.pi * field_parcels["area_m2"]) / (field_parcels["perimeter_m"] ** 2)
    summary_rows.extend(
        [
            {"metric": "Field parcel raster area", "value": parcel_raster_area_m2 / 10_000, "unit": "ha"},
            {"metric": "Field parcel polygon area", "value": field_parcels["area_ha"].sum(), "unit": "ha"},
            {"metric": "Field parcel polygon count", "value": len(field_parcels), "unit": "count"},
            {"metric": "Median parcel size", "value": field_parcels["area_ha"].median(), "unit": "ha"},
            {"metric": "Mean parcel size", "value": field_parcels["area_ha"].mean(), "unit": "ha"},
        ]
    )

summary = pd.DataFrame(summary_rows)
summary_path = QA_DIR / f"cropland_summary_{OUTPUT_TAG}.csv"
summary.to_csv(summary_path, index=False)
summary


,metric,value,unit
0,AOI area,1.740591e+06,ha
1,Cropland pixel area,1.676859e+06,ha
2,Cropland pixel share of AOI,9.633849e-01,fraction


## Parcel Size Distribution

The histogram helps choose sensible parcel filters. Very tiny parcels are usually raster noise or edge fragments.


In [ ]:
if HAS_FIELD_PARCELS:
    fig, ax = plt.subplots(figsize=(10, 6))
    field_parcels["area_ha"].clip(upper=field_parcels["area_ha"].quantile(0.99)).hist(ax=ax, bins=40)
    ax.set_xlabel("Parcel area (ha), clipped at p99 for display")
    ax.set_ylabel("Polygon count")
    ax.set_title("Field parcel size distribution")
    plt.show()

    parcel_stats = field_parcels[["area_ha", "compactness"]].describe()
else:
    parcel_stats = pd.DataFrame()
    print("Skipping parcel size distribution; no field parcel layer is available.")

parcel_stats


Skipping parcel size distribution; no field parcel layer is available.


""


## Filter Noisy Parcels

These thresholds are starting values. Adjust them after visual inspection and local knowledge of field sizes.


In [ ]:
MIN_PARCEL_AREA_HA = 0.05
MAX_PARCEL_AREA_HA = 100.0

if HAS_FIELD_PARCELS:
    cleaned_parcels = field_parcels[
        (field_parcels["area_ha"] >= MIN_PARCEL_AREA_HA)
        & (field_parcels["area_ha"] <= MAX_PARCEL_AREA_HA)
    ].copy()

    flagged_large = field_parcels[field_parcels["area_ha"] > MAX_PARCEL_AREA_HA].copy()
    removed_small = field_parcels[field_parcels["area_ha"] < MIN_PARCEL_AREA_HA].copy()

    cleaned_parcels_path = QA_DIR / f"field_parcels_{OUTPUT_TAG}_cleaned.gpkg"
    cleaned_parcels.to_file(cleaned_parcels_path, layer="field_parcels_cleaned", driver="GPKG")

    if not flagged_large.empty:
        flagged_large.to_file(QA_DIR / f"field_parcels_{OUTPUT_TAG}_large_review.gpkg", layer="large_review", driver="GPKG")

    print(f"Original parcels: {len(field_parcels):,}")
    print(f"Cleaned parcels: {len(cleaned_parcels):,}")
    print(f"Removed small parcels: {len(removed_small):,}")
    print(f"Flagged large parcels: {len(flagged_large):,}")
    print(cleaned_parcels_path)
else:
    cleaned_parcels = gpd.GeoDataFrame({"value": []}, geometry=[], crs=crop_profile["crs"])
    cleaned_parcels_path = None
    print("Skipping parcel cleaning; no field parcel layer is available.")

cleaned_parcels_path


Skipping parcel cleaning; no field parcel layer is available.


## Write Cleaned Parcel Raster

The cleaned raster is aligned to the notebook `02` cropland raster and can be used by later notebooks without re-reading polygons.


In [ ]:
if HAS_FIELD_PARCELS:
    cleaned_shapes = [(geom, 1) for geom in cleaned_parcels.geometry if not geom.is_empty]
    cleaned_raster = rasterize(
        cleaned_shapes,
        out_shape=(crop_profile["height"], crop_profile["width"]),
        transform=crop_profile["transform"],
        fill=0,
        dtype="uint8",
    )

    cleaned_raster_path = QA_DIR / f"field_parcels_{OUTPUT_TAG}_cleaned.tif"
    cleaned_profile = crop_profile.copy()
    cleaned_profile.update(dtype="uint8", count=1, nodata=0, compress="deflate", tiled=True)

    with rasterio.open(cleaned_raster_path, "w", **cleaned_profile) as dst:
        dst.write(cleaned_raster, 1)
else:
    cleaned_raster_path = None
    print("Skipping cleaned parcel raster; no field parcel layer is available.")

cleaned_raster_path


Skipping cleaned parcel raster; no field parcel layer is available.


## Create Validation Sample Points

These points are for manual interpretation or field checking. The three strata are:

- `cropland_parcel`: inside cleaned cropland parcels
- `non_cropland`: inside AOI but outside cleaned parcels
- `edge_uncertain`: near parcel boundaries, where classification is most likely to be ambiguous


In [ ]:
RANDOM_SEED = 42
POINTS_PER_STRATUM = 100
EDGE_BUFFER_M = 60
rng = np.random.default_rng(RANDOM_SEED)


def sample_points_from_mask(mask, count, rng, transform):
    rows = np.argwhere(mask)
    if len(rows) == 0 or count <= 0:
        return []
    selected = rows[rng.choice(len(rows), size=min(count, len(rows)), replace=False)]
    points = []
    for row, col in selected:
        x, y = transform * (float(col) + 0.5, float(row) + 0.5)
        points.append(Point(x, y))
    return points

records = []
if HAS_FIELD_PARCELS and not cleaned_parcels.empty:
    parcel_union = geometry_union(cleaned_parcels.geometry)
    cropland_geom = parcel_union.intersection(aoi_geom)
    non_cropland_geom = aoi_geom.difference(parcel_union)
    edge_geom = geometry_union(cleaned_parcels.boundary.buffer(EDGE_BUFFER_M)).intersection(aoi_geom)

    for stratum, geom in [
        ("cropland_parcel", cropland_geom),
        ("non_cropland", non_cropland_geom),
        ("edge_uncertain", edge_geom),
    ]:
        points = sample_points_in_geometry(geom, POINTS_PER_STRATUM, rng)
        for idx, point in enumerate(points, start=1):
            records.append({"stratum": stratum, "sample_id": f"{stratum}_{idx:03d}", "geometry": point})
else:
    non_crop_mask = ~crop_mask
    for stratum, mask in [
        ("cropland_pixel", crop_mask),
        ("non_cropland_pixel", non_crop_mask),
    ]:
        points = sample_points_from_mask(mask, POINTS_PER_STRATUM, rng, crop_profile["transform"])
        for idx, point in enumerate(points, start=1):
            records.append({"stratum": stratum, "sample_id": f"{stratum}_{idx:03d}", "geometry": point})

validation_points = gpd.GeoDataFrame(records, crs=crop_profile["crs"])
validation_points_path = QA_DIR / f"validation_points_{OUTPUT_TAG}.gpkg"
validation_points_geojson = QA_DIR / f"validation_points_{OUTPUT_TAG}.geojson"
validation_points.to_file(validation_points_path, layer="validation_points", driver="GPKG")
validation_points.to_crs(4326).to_file(validation_points_geojson, driver="GeoJSON")

validation_points.groupby("stratum").size(), validation_points_path, validation_points_geojson


(stratum
 cropland_pixel        100
 non_cropland_pixel    100
 dtype: int64,
 PosixPath('/content/drive/MyDrive/erw_spatial_mrv/data/processed/landcover/validation/jan_jun_2026/validation_points_jan_jun_2026.gpkg'),
 PosixPath('/content/drive/MyDrive/erw_spatial_mrv/data/processed/landcover/validation/jan_jun_2026/validation_points_jan_jun_2026.geojson'))

## Review Validation Points On Imagery


In [ ]:
if SHOW_QA_PLOTS:
    false_color = read_composite(FALSE_COLOR_BANDS)
    fig, ax = plt.subplots(figsize=(12, 10))
    ax.imshow(np.nan_to_num(false_color, nan=0), extent=raster_extent)

    if HAS_FIELD_PARCELS and not cleaned_parcels.empty:
        cleaned_parcels.boundary.plot(ax=ax, color="white", linewidth=0.3)
    else:
        ax.imshow(np.where(crop_mask, 1, np.nan), cmap="summer", alpha=0.35, extent=raster_extent)

    validation_points.plot(ax=ax, column="stratum", markersize=8, legend=True)
    ax.set_title("Validation sample points over false-color Landsat")
    ax.set_axis_off()
    plt.show()
    del false_color
    gc.collect()
else:
    print("Skipping validation point preview. Set SHOW_QA_PLOTS = True to display it.")


Skipping validation point preview. Set SHOW_QA_PLOTS = True to display it.


## Outputs From This Notebook

This notebook writes QA-ready cropland validation products under `data/processed/landcover/validation/jan_jun_2026/`.

Always written:

- `cropland_summary_jan_jun_2026.csv`: AOI area and cropland pixel area summary metrics.
- `validation_points_jan_jun_2026.gpkg`: validation points sampled from cropland and non-cropland pixels when parcel outputs are absent.
- `validation_points_jan_jun_2026.geojson`: the same validation points in web-friendly GeoJSON.

Written only when notebook `02` was run with `GENERATE_FIELD_PARCELS = True`:

- `field_parcels_jan_jun_2026_cleaned.gpkg`: field parcel polygons after removing very small polygons and excluding very large review polygons.
- `field_parcels_jan_jun_2026_cleaned.tif`: cleaned field parcel raster aligned to the notebook `02` cropland raster.
- `field_parcels_jan_jun_2026_large_review.gpkg`: optional large polygons for manual review, written only when large polygons exist.
